# 09 — Final model comparison and ranking

This notebook compares one final representative from each modelling approach while retaining every generated and provided baseline in the repository. It ranks models using the same public five-minute scorer; it does not choose a model from intuition or copy labels from the public ground truth.

The comparison is layered: most approaches share the canonical observations and scenario detector, then differ in how they resolve missing-ball intervals, transitions, probabilities, Viterbi paths, or durations. The intended architecture is `raw observations -> derived observations -> scenario evidence -> anchors -> transition/no-switch hypotheses -> interval resolver -> optional HMM/HSMM/Viterbi smoothing -> H/A/D/X labels`. A model labelled `Kalman` is therefore a Kalman-supported possession policy, not a Kalman possession classifier. The Haiqal row is deliberately outside this shared scenario layer: it is an independent raw-observation distance/speed/dead-ball HMM baseline adapted from `origin/haiqal`, so the comparison can tell whether scenario reasoning adds value. The optional Bayesian-anchor, HSMM, MLE-calibrated, and EM experiments are ranked here only after their outputs are generated and scored on the same evaluation slice.

In [1]:
from pathlib import Path
import os
import subprocess
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
while not (ROOT / 'evaluation').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.environ['MTAH_DATA_DIR'] = str(ROOT / 'data')
RESULTS_DIR = ROOT / 'evaluation' / 'results'
SCOREBOARD_PATH = ROOT / 'evaluation' / 'scoreboard.csv'
PROVIDED_SCOREBOARD_PATH = ROOT / 'evaluation' / 'compare_results' / 'scoreboard.csv'
RERUN_MODELS = False
RANK_BY = ['spell_f1_mean', 'macro_f1', 'trans_f1_25']
SHOW_ALL_GENERATED_RESULTS = False
print(f'RERUN_MODELS={RERUN_MODELS}')
print(f'ranking keys: {RANK_BY}')
print(f'results directory: {RESULTS_DIR}')

RERUN_MODELS=False
ranking keys: ['spell_f1_mean', 'macro_f1', 'trans_f1_25']
results directory: /Users/luke/Desktop/barca-hack-repo-staging/evaluation/results


## Re-run toggle

Leave `RERUN_MODELS = False` to load the committed results immediately. Set it to `True` when the feature or model code has changed; the scenario-aware experiment runners and the independent Haiqal runner will regenerate their outputs, then the common evaluator will rebuild the scoreboard. The runners intentionally retain their full baseline output sets; the ranking table below only presents the final representatives.

In [2]:
RUNNER_MODULES = [
    'possession_lab.run_interpolated_flight_experiment',
    'possession_lab.run_temporal_experiment',
    'possession_lab.run_haiqal_baseline',
    'possession_lab.run_calibration_experiment',
]

if RERUN_MODELS:
    runner_environment = os.environ.copy()
    runner_environment['MTAH_DATA_DIR'] = str(ROOT / 'data')
    for module in RUNNER_MODULES:
        print(f'Running {module} ...')
        completed = subprocess.run(
            [sys.executable, '-m', module],
            cwd=ROOT,
            env=runner_environment,
            capture_output=True,
            text=True,
            check=False,
        )
        print(completed.stdout[-2000:])
        if completed.returncode:
            print(completed.stderr[-4000:])
            raise RuntimeError(f'{module} failed with exit status {completed.returncode}')
    evaluator = subprocess.run(
        [sys.executable, str(ROOT / 'evaluation' / 'run_evaluation.py'), '--gt', 'public', '--no-detail'],
        cwd=ROOT,
        capture_output=True,
        text=True,
        check=False,
    )
    print(evaluator.stdout[-4000:])
    if evaluator.returncode:
        print(evaluator.stderr[-4000:])
        raise RuntimeError(f'public evaluator failed with exit status {evaluator.returncode}')
else:
    print('Loading committed model outputs; no runners were started.')

Loading committed model outputs; no runners were started.


## Final representatives

The manifest is intentionally small and auditable. The four `provided baseline` rows come from the supplied comparison set. The remaining rows are one representative of the current scenario, missing-ball, interval-transition, probability-emission, Viterbi, and HSMM approaches, plus the independent Haiqal raw-observation baseline. Optional Bayesian-anchor and additional duration rows are included when their runner outputs exist. Haiqal does not consume scenario evidence; its row tests the value of the layered architecture against a standalone distance/speed/dead-ball HMM. All other generated CSVs remain available for audit and can be shown with `SHOW_ALL_GENERATED_RESULTS`.

In [3]:
manifest = pd.DataFrame([
    {'submission': 'stage_hq_viterbi', 'approach': 'provided baseline', 'decision_layer': 'provided Viterbi decoder'},
    {'submission': 'evidence_bounded_deadball', 'approach': 'provided baseline', 'decision_layer': 'provided evidence-bounded deadball'},
    {'submission': 'aligned_physics_veto', 'approach': 'provided baseline', 'decision_layer': 'provided aligned physics veto'},
    {'submission': 'protected_aligned_reception', 'approach': 'provided baseline', 'decision_layer': 'provided protected reception'},
    {'submission': 'diagnostic_visible_ball_low_motion_dead_ball', 'approach': 'scenario + flight', 'decision_layer': 'visible-ball low-motion deadball candidate'},
    {'submission': 'baseline_scenario_bidirectional_interpolated_flight_ensemble', 'approach': 'scenario + flight', 'decision_layer': 'bidirectional scenario/interpolated flight ensemble'},
    {'submission': 'baseline_same_team_missing_ball_trajectory', 'approach': 'missing-ball geometry', 'decision_layer': 'same-team trajectory bridge'},
    {'submission': 'baseline_same_team_kalman_missing_ball_trajectory', 'approach': 'missing-ball geometry', 'decision_layer': 'same-team Kalman-supported trajectory bridge'},
    {'submission': 'baseline_transition_interval_hypothesis', 'approach': 'interval transition', 'decision_layer': 'explicit transition interval hypothesis'},
    {'submission': 'baseline_trusted_transition', 'approach': 'interval transition', 'decision_layer': 'trusted transition resolver'},
    {'submission': 'baseline_transition_ledger_viterbi', 'approach': 'Viterbi', 'decision_layer': 'transition-ledger Viterbi'},
    {'submission': 'baseline_probability_transition_ledger_viterbi', 'approach': 'probability + Viterbi', 'decision_layer': 'normalized evidence-emission Viterbi'},
    {'submission': 'baseline_scenario_confidence_duration', 'approach': 'HSMM/duration', 'decision_layer': 'scenario confidence duration decoder'},
    {'submission': 'baseline_restart_type_specific_interpolated_flight', 'approach': 'restart + flight', 'decision_layer': 'restart-type-specific interpolated flight'},
    {'submission': 'baseline_haiqal_raw_viterbi', 'approach': 'independent raw HMM', 'decision_layer': 'Haiqal distance/speed/dead-ball clues + Viterbi'},
])
optional_manifest = pd.DataFrame([
    {'submission': 'baseline_haiqal_evidence_probability_ledger_viterbi', 'approach': 'our pipeline + Haiqal evidence', 'decision_layer': 'Haiqal raw clues + our probability/transition ledger + Viterbi'},
    {'submission': 'baseline_bayesian_hmm', 'approach': 'Bayesian HMM', 'decision_layer': 'Haiqal-style Dirichlet anchor transition prior + Viterbi'},
    {'submission': 'baseline_hmm_anchor', 'approach': 'HMM', 'decision_layer': 'anchor-initialized diagonal-Gaussian HMM'},
    {'submission': 'baseline_hsmm_anchor', 'approach': 'HSMM/duration', 'decision_layer': 'anchor-initialized duration cleanup'},
    {'submission': 'baseline_scenario_hsmm', 'approach': 'HSMM/duration', 'decision_layer': 'scenario-aware semi-Markov decoder'},
])

current_scores = pd.read_csv(SCOREBOARD_PATH).assign(score_source='current runners')
provided_scores = pd.read_csv(PROVIDED_SCOREBOARD_PATH).assign(score_source='provided comparison set')
scores = pd.concat([current_scores, provided_scores], ignore_index=True, sort=False)
scores = scores.drop_duplicates(subset=['submission'], keep='first')
manifest = pd.concat([manifest, optional_manifest[optional_manifest['submission'].isin(scores['submission'])]], ignore_index=True)
ranked = manifest.merge(scores, on='submission', how='left', validate='one_to_one')
missing = ranked.loc[ranked['macro_f1'].isna(), 'submission'].tolist()
if missing:
    raise FileNotFoundError(f'Manifest outputs are not scored: {missing}')
ranked = ranked.sort_values(RANK_BY, ascending=False).reset_index(drop=True)
ranked = ranked.drop(columns=['rank'], errors='ignore')
ranked.insert(0, 'rank', ranked.index + 1)
ranking_columns = ['rank', 'approach', 'decision_layer', 'submission', 'score_source', 'macro_f1', 'spell_f1_mean', 'trans_f1_25', 'f1_H', 'f1_A', 'f1_D']
display(ranked[ranking_columns])

if SHOW_ALL_GENERATED_RESULTS:
    all_columns = [c for c in ['rank', 'submission', 'macro_f1', 'spell_f1_mean', 'trans_f1_25', 'f1_H', 'f1_A', 'f1_D'] if c in current_scores]
    display(current_scores.sort_values(['spell_f1_mean', 'macro_f1'], ascending=False)[all_columns])

,rank,approach,decision_layer,submission,score_source,macro_f1,spell_f1_mean,trans_f1_25,f1_H,f1_A,f1_D
0,1,scenario + flight,visible-ball low-motion deadball candidate,diagnostic_visible_ball_low_motion_dead_ball,current runners,0.888744,0.600000,0.444444,0.909837,0.847238,0.909156
1,2,restart + flight,restart-type-specific interpolated flight,baseline_restart_type_specific_interpolated_fl...,current runners,0.656092,0.573333,0.181818,0.826585,0.808169,0.333522
2,3,provided baseline,provided evidence-bounded deadball,evidence_bounded_deadball,provided comparison set,0.773604,0.567742,0.333333,0.858487,0.830460,0.631866
3,4,Viterbi,transition-ledger Viterbi,baseline_transition_ledger_viterbi,current runners,0.772457,0.567742,0.333333,0.857893,0.827611,0.631866
4,5,interval transition,explicit transition interval hypothesis,baseline_transition_interval_hypothesis,current runners,0.771958,0.567742,0.333333,0.856709,0.827301,0.631866
5,6,interval transition,trusted transition resolver,baseline_trusted_transition,current runners,0.771958,0.567742,0.333333,0.856709,0.827301,0.631866
6,7,probability + Viterbi,normalized evidence-emission Viterbi,baseline_probability_transition_ledger_viterbi,current runners,0.771958,0.567742,0.333333,0.856709,0.827301,0.631866
7,8,HSMM/duration,scenario confidence duration decoder,baseline_scenario_confidence_duration,current runners,0.771958,0.567742,0.333333,0.856709,0.827301,0.631866
8,9,our pipeline + Haiqal evidence,Haiqal raw clues + our probability/transition ...,baseline_haiqal_evidence_probability_ledger_vi...,current runners,0.771958,0.567742,0.333333,0.856709,0.827301,0.631866
9,10,scenario + flight,bidirectional scenario/interpolated flight ens...,baseline_scenario_bidirectional_interpolated_f...,current runners,0.671635,0.567742,0.171429,0.829347,0.808511,0.377049


## Haiqal evidence ablation

This compares the independent Haiqal output with the current probability/transition-ledger path and the combined variant. The combined variant imports only Haiqal's raw distance, speed, visibility, out-of-pitch, and dead-ball clues; it does not import Haiqal's final labels or replace our resolver. Identical scores are evidence that the added clues are redundant or underweighted, not evidence that the experiment was skipped.

In [4]:
ablation_names = [
    'baseline_haiqal_raw_viterbi',
    'baseline_probability_transition_ledger_viterbi',
    'baseline_haiqal_evidence_probability_ledger_viterbi',
    'diagnostic_visible_ball_low_motion_dead_ball',
]
ablation_columns = [c for c in ['submission', 'macro_f1', 'spell_f1_mean', 'trans_f1_25'] if c in current_scores]
display(current_scores[current_scores['submission'].isin(ablation_names)][ablation_columns].sort_values('submission'))

,submission,macro_f1,spell_f1_mean,trans_f1_25
13,baseline_haiqal_evidence_probability_ledger_vi...,0.771958,0.567742,0.333333
69,baseline_haiqal_raw_viterbi,0.767720,0.514286,0.266667
16,baseline_probability_transition_ledger_viterbi,0.771958,0.567742,0.333333
0,diagnostic_visible_ball_low_motion_dead_ball,0.888744,0.600000,0.444444


## Calibration diagnostics (reported separately from the model ranking)

MLE temperature scaling is not a possession submission and is not ranked by macro-F1/spell-F1. It is evaluated with log loss and Brier score on a disjoint public holdout. A worse holdout score is useful evidence of overfitting or distribution shift, not a reason to hide the experiment.

In [5]:
calibration_path = ROOT / 'evaluation' / 'reports' / 'probability_calibration_public_holdout.csv'
if calibration_path.exists():
    display(pd.read_csv(calibration_path))
else:
    print('Calibration report is absent; set RERUN_MODELS=True to generate it.')

,split,model,temperature,log_loss,brier
0,holdout,rule_evidence_softmax,1.000000,1.366923,0.699768
1,holdout,mle_temperature_scaled,0.778684,1.523411,0.741228


## How to interpret the approaches

- Scenario detection is shared evidence for most current models; it is not something to exclude when testing transitions.
- The probability model is a normalized evidence-emission variant, not calibrated probability and not a replacement for scenario detection.
- MLE temperature scaling is supervised calibration only when fitted on a declared reviewed-label split; it must be scored on a disjoint holdout.
- Haiqal's Bayesian-anchor model uses a Dirichlet-smoothed transition prior from algorithmic anchors, not public labels; it is a useful transition-prior comparison, not calibrated softmax output.
- EM is an unsupervised latent-state experiment and must be checked for label switching, anchor violations, and stability before promotion.
- Viterbi is a path smoother over those emissions and transition hypotheses; it is not the primary truth-maker.
- HSMM adds duration assumptions; it remains a comparison approach until duration distributions are learned from reviewed labels.
- Kalman is tested as missing-ball geometry support. Its score answers whether the bridge helps the complete policy, not whether Kalman alone labels possession.
- Rankings on the five-minute public slice are evidence, not universal truth; spell and transition scores are especially noisy on this short benchmark.